In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================
# CELL 1: NATIVE PYTHON 3.10 INSTALLER (V3)
# ==========================================
from google.colab import drive
drive.mount('/content/drive')

import os
import json

print("1. Installing Native Python 3.10 (Bypassing Colab & Conda)...")
!apt-get update -y -q
!apt-get install python3.10 python3.10-dev python3.10-distutils -y -q
!wget -q https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py -q

print("2. Installing PyTorch 2.2.1 and Instant Mamba Wheels...")
!python3.10 -m pip install torch==2.2.1 torchvision==0.17.1 --index-url https://download.pytorch.org/whl/cu121 -q
!python3.10 -m pip install https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.2.0.post2/causal_conv1d-1.2.0.post2+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl -q
!python3.10 -m pip install https://github.com/state-spaces/mamba/releases/download/v1.2.0.post1/mamba_ssm-1.2.0.post1+cu122torch2.2cxx11abiFALSE-cp310-cp310-linux_x86_64.whl -q

print("3. Pre-installing Build Tools & MedPy...")
!python3.10 -m pip install --upgrade pip setuptools wheel pytest-runner numpy cython -q
!python3.10 -m pip install medpy --no-build-isolation -q

print("4. Downloading and Patching UU-Mamba...")
%cd /content
!rm -rf UU-Mamba
!git clone https://github.com/tiffany9056/UU-Mamba.git
!find /content/UU-Mamba/uumamba/nnunetv2 -type f -name "*.py" -exec sed -i 's/architectures.residual_unet/architectures.unet/g' {} +
%cd /content/UU-Mamba/uumamba

print("5. Installing Frameworks (with strict version controls)...")
# Pin MONAI to 1.3.0 so it doesn't crash PyTorch
!python3.10 -m pip install dynamic-network-architectures monai==1.3.0 nibabel skan networkx scikit-image pandas scipy -q
# Block pip's secret build isolation
!python3.10 -m pip install -e . --no-build-isolation -q

print("6. Setting Environment Variables...")
os.environ["nnUNet_raw"] = "/content/drive/MyDrive/COROnet_Drive/Mamba_Helper/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/content/drive/MyDrive/COROnet_Drive/Mamba_Helper"
os.environ["nnUNet_results"] = "/content/drive/MyDrive/COROnet_Drive/Mamba_Helper/nnUNet_results"

print("7. Bypassing PyTorch Security Blocks...")
file_path = "/content/UU-Mamba/uumamba/nnunetv2/inference/predict_from_raw_data.py"
with open(file_path, "r") as f:
    content = f.read()
target = "map_location=torch.device('cpu'))"
replacement = "map_location=torch.device('cpu'), weights_only=False)"
if "weights_only=False" not in content:
    content = content.replace(target, replacement)
    with open(file_path, "w") as f:
        f.write(content)

print("\n🚀 NATIVE PYTHON 3.10 ENVIRONMENT LOCKED AND LOADED!")

ValueError: Mountpoint must not already contain files

In [ ]:
import os
import glob

print("Scanning ICC_Data for nnU-Net formatting...")
input_dir = "/content/drive/MyDrive/COROnet_Drive/ICC_Data"
files = glob.glob(os.path.join(input_dir, "*.nii.gz"))

fixed_count = 0
for file_path in files:
    # Check if it already has the required nnU-Net suffix
    if not file_path.endswith("_0000.nii.gz"):
        # Snap the _0000 tag onto the end of the filename
        new_path = file_path.replace(".nii.gz", "_0000.nii.gz")
        os.rename(file_path, new_path)
        fixed_count += 1

print(f"✅ Fixed and renamed {fixed_count} files! Total ready files: {len(files)}")

Scanning ICC_Data for nnU-Net formatting...
✅ Fixed and renamed 0 files! Total ready files: 20


In [ ]:
# ==========================================
# Phase 1.5: Multi-Label Heart Chamber Cropper
# ==========================================
import os
import glob
import nibabel as nib
import numpy as np

# 1. Paths
DRIVE_ROOT = '/content/drive/MyDrive/COROnet_Drive'
RAW_CT_DIR = os.path.join(DRIVE_ROOT, "ICC_Data")
HEART_MASK_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Helper/COROnet_Heart")
CROPPED_CT_DIR = os.path.join(DRIVE_ROOT, "Segmentation/Cropped_Data")

os.makedirs(CROPPED_CT_DIR, exist_ok=True)

raw_files = sorted(glob.glob(os.path.join(RAW_CT_DIR, "*.nii.gz")))
PADDING = 25

# Target Labels: 1 (LA), 2 (LV), 3 (RA), 4 (RV)
TARGET_CHAMBERS = [1, 2, 3, 4]

print(f"✂️ Starting Multi-Label Cropping for {len(raw_files)} patients...\n")

for raw_path in raw_files:
    filename = os.path.basename(raw_path)
    patient_id = filename.replace("_0000.nii.gz", "").replace(".nii.gz", "")
    cropped_save_path = os.path.join(CROPPED_CT_DIR, filename)

    if os.path.exists(cropped_save_path):
        print(f"[{patient_id}] ⏭️ Skipping (Already cropped)")
        continue

    # 2. Load the multi-label heart file
    # This assumes the file is at: .../Segmentation/Helper/COROnet_Heart/patient_id/heart.nii.gz
    heart_mask_path = os.path.join(HEART_MASK_DIR, patient_id, "heart.nii.gz")

    if not os.path.exists(heart_mask_path):
        print(f"⚠️ Warning: heart.nii.gz not found for {patient_id}. Skipping.")
        continue

    try:
        mask_img = nib.load(heart_mask_path)
        mask_data = mask_img.get_fdata()

        # 3. Filter for specific chambers (LA, LV, RA, RV)
        # We find coordinates where the label is 1, 2, 3, or 4
        chamber_mask = np.isin(mask_data, TARGET_CHAMBERS)
        coords = np.argwhere(chamber_mask)

        if len(coords) == 0:
            print(f"🚨 Error: Could not find chambers 1-4 in heart.nii.gz for {patient_id}.")
            continue

        # 4. Calculate Bounding Box
        min_x, min_y, min_z = coords.min(axis=0)
        max_x, max_y, max_z = coords.max(axis=0)

        # 5. Load Raw CT and Apply Padding
        raw_img = nib.load(raw_path)
        shape = raw_img.shape

        min_x, max_x = max(0, min_x - PADDING), min(shape[0], max_x + PADDING)
        min_y, max_y = max(0, min_y - PADDING), min(shape[1], max_y + PADDING)
        min_z, max_z = max(0, min_z - PADDING), min(shape[2], max_z + PADDING)

        # 6. Execute and Save the Crop
        cropped_img = raw_img.slicer[int(min_x):int(max_x), int(min_y):int(max_y), int(min_z):int(max_z)]
        nib.save(cropped_img, cropped_save_path)

        print(f"✅ {patient_id}: Cropped to chambers. Shape: {cropped_img.shape}")

    except Exception as e:
        print(f"🚨 Error processing {patient_id}: {e}")

print("\n🎉 Heart-ROI Extraction Complete!")

✂️ Starting Multi-Label Cropping for 20 patients...

✅ ICC_Patient_0169: Cropped to chambers. Shape: (329, 258, 378)
✅ ICC_Patient_0170: Cropped to chambers. Shape: (358, 252, 351)
✅ ICC_Patient_0171: Cropped to chambers. Shape: (252, 228, 437)
✅ ICC_Patient_0172: Cropped to chambers. Shape: (369, 280, 454)
✅ ICC_Patient_0173: Cropped to chambers. Shape: (293, 288, 402)
✅ ICC_Patient_0174: Cropped to chambers. Shape: (350, 270, 340)
✅ ICC_Patient_0175: Cropped to chambers. Shape: (279, 229, 400)
✅ ICC_Patient_0176: Cropped to chambers. Shape: (320, 259, 383)
✅ ICC_Patient_0177: Cropped to chambers. Shape: (303, 258, 452)
✅ ICC_Patient_0178: Cropped to chambers. Shape: (328, 281, 503)
✅ ICC_Patient_0179: Cropped to chambers. Shape: (362, 255, 436)
✅ ICC_Patient_0180: Cropped to chambers. Shape: (322, 300, 436)
✅ ICC_Patient_0181: Cropped to chambers. Shape: (357, 255, 355)
✅ ICC_Patient_0182: Cropped to chambers. Shape: (306, 231, 399)
✅ ICC_Patient_0183: Cropped to chambers. Shape: (36

In [ ]:
!pip install "numpy<2.0"
# Fix the Transformers version
!pip install "transformers<4.40.0" -q

# Repair the fractured PyTorch environment
!pip install torch==2.2.1 torchvision==0.17.1 torchaudio==2.2.1 --index-url https://download.pytorch.org/whl/cu121 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 27.0 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [ ]:
import os
import shutil


# 1. Setup Directories
# We predict to a LOCAL folder first to avoid Google Drive lag
LOCAL_PREDS = "/content/local_preds"
DRIVE_FINAL = "/content/drive/MyDrive/COROnet_Drive/Segmentation/Mamba_Coronary_Arteries"

os.makedirs(LOCAL_PREDS, exist_ok=True)
os.makedirs(DRIVE_FINAL, exist_ok=True)

# 2. Workspace Bridges (Standard Setup)
!rm -rf /workspace/UU-Mamba/data/*
!mkdir -p /workspace/UU-Mamba/data
!ln -s /content/drive/MyDrive/COROnet_Drive/Mamba_Helper/nnUNet_raw /workspace/UU-Mamba/data/nnUNet_raw
!ln -s /content/drive/MyDrive/COROnet_Drive/Mamba_Helper/workspace/UU-Mamba/data/nnUNet_preprocessed
!ln -s /content/drive/MyDrive/COROnet_Drive/Mamba_Helper/nnUNet_results /workspace/UU-Mamba/data/nnUNet_results

print(f"🚀 Launching Local Inference (Bypassing Drive Latency)...")

# 3. Run Inference to LOCAL Disk
# We use more workers (-npp 4 -nps 4) because Local Disk can handle the speed!
!export MPLBACKEND="agg" && \
 export nnUNet_raw="/workspace/UU-Mamba/data/nnUNet_raw" && \
 export nnUNet_preprocessed="/workspace/UU-Mamba/data/nnUNet_preprocessed" && \
 export nnUNet_results="/workspace/UU-Mamba/data/nnUNet_results" && \
 nnUNetv2_predict \
 -i /content/drive/MyDrive/COROnet_Drive/Segmentation/Cropped_Data \
 -o {LOCAL_PREDS} \
 -d Dataset101_ImageCAS \
 -c 3d_fullres \
 -f all \
 -tr nnUNetTrainerUMambaEnc \
 --disable_tta \
 -chk checkpoint_ImageCAS.pth \
 -npp 4 -nps 4

# 4. Sync Local Results to Google Drive
print(f"\n📂 Syncing predictions to Google Drive...")
files = os.listdir(LOCAL_PREDS)
for f in files:
    src = os.path.join(LOCAL_PREDS, f)
    dst = os.path.join(DRIVE_FINAL, f)
    if not os.path.exists(dst):
        shutil.copy(src, dst)
        print(f" ✅ Synced: {f}")

print("\n🎉 ALL PATIENTS COMPLETED AND SYNCED!")

🚀 Launching Local Inference (Bypassing Drive Latency)...

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

There are 20 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 20 cases that I would like to predict

Predicting ICC_Patient_0169:
perform_everything_on_device: True
100% 45/45 [00:23<00:00,  1.91it/s]
sending off prediction to background worker for resampling and export
done with ICC_Patient_0169

Predicting ICC_Patient_0170:
perform_everything_on_device: True
100% 60/60 [00:31<00:00,  1.89it/s]
sending off prediction to background worker for resampling and export
done with 

In [ ]:
import os
import glob
import nibabel as nib
import numpy as np
from skimage.morphology import skeletonize, ball
from scipy.ndimage import binary_dilation

# --- FOLDERS ---
INPUT_ROOT = "/content/drive/MyDrive/COROnet_Drive/Segmentation/Helper/Mamba_Coronary_Arteries"
OUTPUT_ROOT = "/content/drive/MyDrive/COROnet_Drive/Segmentation/Helper/Mamba_Coronary_Arteries_Refined"

os.makedirs(OUTPUT_ROOT, exist_ok=True)

# Set your desired thickness radius in voxels (e.g., radius=2 means a tube ~5 voxels wide)
THICKNESS_RADIUS = 6

# Find all prediction masks from the previous nnU-Net step
coronary_masks = glob.glob(os.path.join(INPUT_ROOT, "*.nii.gz"))

print(f"🔍 Found {len(coronary_masks)} coronary masks. Starting Skeletonization & Thickening...\n")

for i, mask_path in enumerate(coronary_masks):
    filename = os.path.basename(mask_path)
    patient_id = filename.replace(".nii.gz", "")

    # Define output file paths in the new Refined directory
    skel_out = os.path.join(OUTPUT_ROOT, f"{patient_id}_skeleton.nii.gz")
    thick_out = os.path.join(OUTPUT_ROOT, f"{patient_id}_thickened_r{THICKNESS_RADIUS}.nii.gz")

    if os.path.exists(thick_out):
        print(f"[{i+1}/{len(coronary_masks)}] ⏭️  Skipping {patient_id} (Already thickened)")
        continue

    print(f"[{i+1}/{len(coronary_masks)}] 🦴 Processing {patient_id}...")

    try:
        # 1. Load the NIfTI file
        img = nib.load(mask_path)
        img_data = img.get_fdata()

        # Ensure it's a binary mask
        binary_mask = (img_data > 0).astype(np.uint8)

        # 2. SKELETONIZATION (Find the centerlines)
        # This reduces the complex shape down to a 1-voxel wide tree
        skeleton = skeletonize(binary_mask)

        # 3. RECONSTRUCTION / THICKENING
        # Create a 3D spherical structuring element ('ball') of the desired radius
        structuring_element = ball(THICKNESS_RADIUS)

        # Dilate the skeleton using the 3D ball to create uniform tubes
        thickened = binary_dilation(skeleton, structure=structuring_element)

        # 4. Save the results back to Drive
        # Save Skeleton
        skel_img = nib.Nifti1Image(skeleton.astype(np.uint8), img.affine, img.header)
        nib.save(skel_img, skel_out)

        # Save Thickened Reconstruction
        thick_img = nib.Nifti1Image(thickened.astype(np.uint8), img.affine, img.header)
        nib.save(thick_img, thick_out)

        print(f"   ✅ Saved skeleton and thickened (radius={THICKNESS_RADIUS}) mask.")

    except Exception as e:
        print(f"   🚨 Error processing {patient_id}: {e}")

print("\n🎉 ALL POST-PROCESSING COMPLETE!")

🔍 Found 20 coronary masks. Starting Skeletonization & Thickening...

[1/20] 🦴 Processing ICC_Patient_0181...
   ✅ Saved skeleton and thickened (radius=6) mask.
[2/20] 🦴 Processing ICC_Patient_0170...
   ✅ Saved skeleton and thickened (radius=6) mask.
[3/20] 🦴 Processing ICC_Patient_0184...
   ✅ Saved skeleton and thickened (radius=6) mask.
[4/20] 🦴 Processing ICC_Patient_0182...
   ✅ Saved skeleton and thickened (radius=6) mask.
[5/20] 🦴 Processing ICC_Patient_0173...
   ✅ Saved skeleton and thickened (radius=6) mask.
[6/20] 🦴 Processing ICC_Patient_0176...
   ✅ Saved skeleton and thickened (radius=6) mask.
[7/20] 🦴 Processing ICC_Patient_0172...
   ✅ Saved skeleton and thickened (radius=6) mask.
[8/20] 🦴 Processing ICC_Patient_0177...
   ✅ Saved skeleton and thickened (radius=6) mask.
[9/20] 🦴 Processing ICC_Patient_0175...
   ✅ Saved skeleton and thickened (radius=6) mask.
[10/20] 🦴 Processing ICC_Patient_0185...
   ✅ Saved skeleton and thickened (radius=6) mask.
[11/20] 🦴 Processing

In [ ]:
import os
import glob
import shutil

# --- FOLDERS ---
REFINED_DIR = "/content/drive/MyDrive/COROnet_Drive/Segmentation/Helper/Mamba_Coronary_Arteries_Refined"
CLASSIFIER_INPUT_DIR = "/content/drive/MyDrive/COROnet_Drive/Segmentation/Coronary_Mapping_Input"

os.makedirs(CLASSIFIER_INPUT_DIR, exist_ok=True)

# Find all thickened files
thickened_files = glob.glob(os.path.join(REFINED_DIR, "*_thickened_*.nii.gz"))

print(f"📁 Found {len(thickened_files)} thickened masks. Copying and renaming to Classifier Input folder...\n")

for file_path in thickened_files:
    filename = os.path.basename(file_path)

    # Extract patient ID (e.g., ICC_Patient_0181 from ICC_Patient_0181_thickened_r6.nii.gz)
    patient_id = filename.split("_thickened_")[0]
    new_filename = f"{patient_id}.nii.gz"

    new_path = os.path.join(CLASSIFIER_INPUT_DIR, new_filename)

    # Copy file to new directory with the cleaned name
    shutil.copy(file_path, new_path)
    print(f"✅ Copied: {filename} -> {new_filename}")

print("\n🎉 All files successfully prepared for the classifier!")

📁 Found 20 thickened masks. Copying and renaming to Classifier Input folder...

✅ Copied: ICC_Patient_0181_thickened_r6.nii.gz -> ICC_Patient_0181.nii.gz
✅ Copied: ICC_Patient_0170_thickened_r6.nii.gz -> ICC_Patient_0170.nii.gz
✅ Copied: ICC_Patient_0184_thickened_r6.nii.gz -> ICC_Patient_0184.nii.gz
✅ Copied: ICC_Patient_0182_thickened_r6.nii.gz -> ICC_Patient_0182.nii.gz
✅ Copied: ICC_Patient_0173_thickened_r6.nii.gz -> ICC_Patient_0173.nii.gz
✅ Copied: ICC_Patient_0176_thickened_r6.nii.gz -> ICC_Patient_0176.nii.gz
✅ Copied: ICC_Patient_0172_thickened_r6.nii.gz -> ICC_Patient_0172.nii.gz
✅ Copied: ICC_Patient_0177_thickened_r6.nii.gz -> ICC_Patient_0177.nii.gz
✅ Copied: ICC_Patient_0175_thickened_r6.nii.gz -> ICC_Patient_0175.nii.gz
✅ Copied: ICC_Patient_0185_thickened_r6.nii.gz -> ICC_Patient_0185.nii.gz
✅ Copied: ICC_Patient_0178_thickened_r6.nii.gz -> ICC_Patient_0178.nii.gz
✅ Copied: ICC_Patient_0187_thickened_r6.nii.gz -> ICC_Patient_0187.nii.gz
✅ Copied: ICC_Patient_0183_thick